# Local RAG Assistant for Customer Support Ticket Triage

Runs entirely on your Mac: **MiniLM** finds similar past tickets, **Llama 3.2 (via Ollama)** writes the triage.

Run the cells top to bottom. The first run downloads the MiniLM model (~80 MB) once.
Make sure Ollama is running before the generation cells (it listens on `localhost:11434`).

## 1. Configuration

In [ ]:
import os, glob, time, json
import pandas as pd

DATA_DIR    = "data"                    # folder that holds your Kaggle CSV
EMBED_MODEL = "all-MiniLM-L6-v2"        # embedding model (downloads on first use)
GEN_MODEL   = "llama3.2:3b"             # local model served by Ollama
OLLAMA_URL  = "http://localhost:11434/api/generate"
K           = 3                          # how many past tickets to retrieve
KB_SIZE     = 1500                       # how many tickets go in the knowledge base

## 2. Load the dataset and keep the English tickets

In [ ]:
# Picks the largest CSV in data/ (so it uses the full file, not the small sample)
csvs = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")), key=os.path.getsize, reverse=True)
print("CSV files found:", csvs)

df = pd.read_csv(csvs[0])
print("Total rows:", len(df))

df = df[df["language"] == "en"].copy()
df = df.dropna(subset=["body", "answer"]).reset_index(drop=True)
print("English rows with body + answer:", len(df))
df[["subject", "queue", "type", "priority"]].head()

## 3. Build the knowledge base

A subset of past tickets. We search on the customer's problem (`body`) and later show the agent's `answer` as guidance.

In [ ]:
kb = df.sample(n=min(KB_SIZE, len(df)), random_state=42).reset_index(drop=True)
kb_texts = (kb["subject"].fillna("") + ". " + kb["body"]).tolist()
print("Knowledge base size:", len(kb))

## 4. Embed the knowledge base with MiniLM

First run downloads the model (~80 MB), then it's cached.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
kb_emb = embedder.encode(kb_texts, convert_to_numpy=True, show_progress_bar=True).astype("float32")
print("Embeddings shape:", kb_emb.shape)   # (KB_SIZE, 384)

## 5. Build the FAISS index

Cosine similarity via inner product on normalized vectors.

In [ ]:
import faiss

faiss.normalize_L2(kb_emb)
index = faiss.IndexFlatIP(kb_emb.shape[1])
index.add(kb_emb)
print("Indexed vectors:", index.ntotal)

## 6. Retrieval: find the K most similar past tickets

In [ ]:
def retrieve(query, k=K):
    q = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    scores, idx = index.search(q, k)
    hits = []
    for rank, (i, s) in enumerate(zip(idx[0], scores[0]), start=1):
        row = kb.iloc[int(i)]
        hits.append({"n": rank, "score": float(s),
                     "subject": row["subject"], "body": row["body"],
                     "answer": row["answer"], "queue": row["queue"],
                     "type": row["type"], "priority": row["priority"]})
    return hits

## 7. Build a grounded, cited prompt

In [ ]:
def build_prompt(ticket_text, hits):
    context = ""
    for h in hits:
        context += f"[{h['n']}] (queue: {h['queue']}, type: {h['type']}, priority: {h['priority']})\n"
        context += f"Customer: {h['body']}\n"
        context += f"Agent answer: {h['answer']}\n\n"
    prompt = (
        "You are a support triage assistant. Using ONLY the past tickets below, "
        "triage the NEW ticket. Cite the past tickets you use with [1], [2], etc. "
        "If the past tickets do not cover it, say there is not enough information.\n\n"
        f"PAST TICKETS:\n{context}"
        f"NEW TICKET:\n{ticket_text}\n\n"
        "Answer with: Suggested queue, Type, Priority, Draft first response, and Citations."
    )
    return prompt

## 8. Call the local model through Ollama

Ollama must be running (`ollama serve` happens automatically when the app is open).

In [ ]:
import requests

def ask_ollama(prompt, model=GEN_MODEL):
    t0 = time.time()
    r = requests.post(OLLAMA_URL, json={"model": model, "prompt": prompt, "stream": False})
    r.raise_for_status()
    return r.json()["response"], time.time() - t0

## 9. Try it on one incoming ticket

We take a random English ticket as the "new" one so you can compare the model's triage against the real labels.

In [ ]:
sample = df.sample(1, random_state=7).iloc[0]
incoming = f"{sample['subject']}. {sample['body']}"

print("INCOMING TICKET:\n", incoming[:400], "\n")
print("TRUE labels -> queue:", sample["queue"], "| type:", sample["type"], "| priority:", sample["priority"], "\n")

hits = retrieve(incoming)
answer, secs = ask_ollama(build_prompt(incoming, hits))
print("=== MODEL TRIAGE (with RAG) ===")
print(answer)
print(f"\n(generated locally in {secs:.1f}s)")

## 10. Baseline: same model, NO retrieval

This is the baseline for Section 9 of your proposal. Compare its triage to the RAG one above.

In [ ]:
baseline_prompt = (
    "You are a support triage assistant. Triage this ticket. "
    "Give Suggested queue, Type, Priority, and a Draft first response.\n\n"
    f"NEW TICKET:\n{incoming}"
)
base_answer, base_secs = ask_ollama(baseline_prompt)
print("=== BASELINE TRIAGE (no retrieval) ===")
print(base_answer)
print(f"\n(baseline, {base_secs:.1f}s)")

## 11. Try your own ticket text

In [ ]:
my_ticket = "My VPN keeps disconnecting every few minutes since the last update."

hits = retrieve(my_ticket)
answer, secs = ask_ollama(build_prompt(my_ticket, hits))
print(answer)
print(f"\n({secs:.1f}s)")

# See which past tickets were retrieved:
for h in hits:
    print(f"\n[{h['n']}] score={h['score']:.2f} | {h['queue']} / {h['type']} / {h['priority']}")
    print("   ", h['subject'])